# Week 8: Images as Numbers: Handwritten Digits

**Grade band:** 6 to 8  |  **Duration:** 60 minutes  |  **Platform:** JupyterLite (browser, no account) or Google Colab

**How to use this notebook:** run each cell from top to bottom with Shift + Enter. Read the text, run the code, then complete the challenge cells marked **SOLUTION**. Save your work at the end of the session (File > Download) so it can be uploaded to your portfolio.

> **Teacher copy.** This notebook contains completed answers for every tier. Do not distribute to students. Use it to check work against the validation checklist.

## Hook: Teach a machine in 60 seconds

Your teacher will open Teachable Machine (no code) and train it on two hand gestures with the webcam. It works in a minute. But *how*? Today you build the same idea from scratch with real numbers you can see.

In [ ]:
# Setup: run this cell first.
import pandas as pd
import matplotlib.pyplot as plt

# If a data file is not found next to this notebook (for example on Google Colab),
# it is loaded from the Wize data folder online instead. Replace this URL after publishing.
DATA_URL = "https://raw.githubusercontent.com/wizeacademy/ml-ai-6-8/main/notebooks/data/"

def load(name):
    """Load a Wize dataset by file name, from the local data folder or from the web."""
    try:
        return pd.read_csv("data/" + name)
    except Exception:
        return pd.read_csv(DATA_URL + name)

print("Setup complete. pandas and matplotlib are ready.")

## Teach 1: An image is a grid of numbers

scikit-learn ships with 1,797 tiny handwritten digits. Each one is an 8 by 8 grid of pixels, and each pixel is a number from 0 (white) to 16 (black ink). Sixty four numbers per image. That is all a computer ever sees.

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()

print("Number of images:", len(digits.images))
print("Shape of one image:", digits.images[0].shape)
print("The first image as numbers:")
print(digits.images[0].astype(int))
print("Its label:", digits.target[0])

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, image, label in zip(axes.ravel(), digits.images[:16], digits.target[:16]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("Sixteen digits and their labels")
plt.show()

## Teach 2: Same recipe, new data

Flatten each 8 by 8 grid into a row of 64 features. Then it is just a table, and KNN works exactly like it did on penguins. **Concept checkpoint:** which two digits do you think will be confused most?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

X = digits.data          # 1797 rows, 64 columns (already flattened)
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)
knn = KNeighborsClassifier(n_neighbors=3).fit(X_train, y_train)
preds = knn.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, preds) * 100, 1), "%")

ConfusionMatrixDisplay.from_predictions(y_test, preds, cmap="Blues")
plt.title("Which digits get confused?")
plt.show()

## SOLUTION: Challenge

**Timer suggestion: 25 minutes.**

### Mild
Show the first 8 **test** images with two titles each: the real label and the prediction. Find one the model got wrong (change the slice if the first 8 are all correct).

### Medium
Train a `DecisionTreeClassifier` on the same data and compare its accuracy to KNN. Which model wins on images, and by how much?

### Spicy
Make the images worse on purpose. Add random noise to the test images (`X_test + np.random.normal(0, 4, X_test.shape)`) and measure accuracy again. Then try noise of 8 and 12. Plot accuracy vs noise. What does this tell you about real world cameras?

In [ ]:
# MILD: real vs predicted
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"real {y_test[i]}\npred {preds[i]}", fontsize=9)
    ax.axis("off")
plt.show()

import numpy as np
wrong = np.where(preds != y_test)[0]
print("Number of mistakes:", len(wrong))
if len(wrong):
    i = wrong[0]
    plt.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    plt.title(f"A mistake: real {y_test[i]}, predicted {preds[i]}")
    plt.axis("off")
    plt.show()

In [ ]:
# MEDIUM: decision tree vs KNN
from sklearn.tree import DecisionTreeClassifier
tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
tree_acc = accuracy_score(y_test, tree.predict(X_test))
knn_acc = accuracy_score(y_test, preds)
print("KNN accuracy:          ", round(knn_acc * 100, 1), "%")
print("Decision tree accuracy:", round(tree_acc * 100, 1), "%")
print("KNN wins by", round((knn_acc - tree_acc) * 100, 1), "points. Trees split on one pixel at a time, which is a poor fit for images.")

In [ ]:
# SPICY: noisy images
import numpy as np
np.random.seed(0)
noise_levels = [0, 4, 8, 12]
accs = []
for level in noise_levels:
    noisy = X_test + np.random.normal(0, level, X_test.shape)
    accs.append(accuracy_score(y_test, knn.predict(noisy)))
    print("noise", level, "accuracy", round(accs[-1] * 100, 1), "%")

plt.plot(noise_levels, accs, marker="o", color="#f7941d")
plt.title("Accuracy drops as images get noisier")
plt.xlabel("Noise level")
plt.ylabel("Accuracy")
plt.show()

plt.imshow((X_test[0] + np.random.normal(0, 12, 64)).reshape(8, 8), cmap="gray_r")
plt.title("What the model sees at noise 12")
plt.axis("off")
plt.show()

## Extra activities (if you finish early)

- Print `digits.images[5]` as numbers and try to read the digit from the numbers alone.
- Which digit has the highest accuracy? Which the lowest? Use the confusion matrix.
- Explain to a partner why a self driving car's camera in the rain is like the Spicy challenge.

## Reflection

- How is an image like the penguin table from week 4?
- What surprised you about how a computer "sees"?